# Phishing Website Detection — Complete CRISP-DM Walkthrough


---

## What this notebook covers

1. **Business Understanding** — what problem we are solving and why it matters  
2. **Data Understanding** — what the datasets contain and what the features mean  
3. **Data Preparation** — how the raw data is cleaned and made usable  
4. **Modeling** — how several machine-learning models are trained and compared  
5. **Evaluation** — how we check whether the results are realistic and not misleading  
6. **Deployment** — how the final phishing prototype can be run as a simple app


## CRISP-DM roadmap 

**CRISP-DM** is a standard project structure for data science work:

- **Business Understanding**: define the real-world goal
- **Data Understanding**: inspect the datasets and understand the variables
- **Data Preparation**: clean the data and remove problematic columns
- **Modeling**: train different models and compare them fairly
- **Evaluation**: test whether the model truly generalises
- **Deployment**: turn the model into something practical, such as a small app

In this project, the real-world question is:

> **Can we accurately detect phishing websites and explain why the model makes its predictions?**


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path(r"C:/Users/nanaa/OneDrive/Naive Bayes Nana Anderson")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        str(c).strip().replace("ï»¿", "").replace("\ufeff", "").strip(";")
        for c in df.columns
    ]
    return df


def load_repaired_csv(csv_path: Path) -> pd.DataFrame:
    with open(csv_path, "r", encoding="utf-8", errors="ignore") as file:
        lines = [line.rstrip("\n") for line in file if line.strip()]

    headers = [part.strip().strip(";") for part in lines[0].split(",")]
    expected_fields = len(headers)
    repaired_rows = []

    for line in lines[1:]:
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < expected_fields:
            continue
        if len(parts) > expected_fields:
            extra_parts = len(parts) - expected_fields + 1
            parts = [",".join(parts[:extra_parts])] + parts[extra_parts:]
        repaired_rows.append(parts[:expected_fields])

    repaired_df = pd.DataFrame(repaired_rows, columns=headers)
    repaired_df = clean_column_names(repaired_df)

    for col in repaired_df.select_dtypes(include="object").columns:
        repaired_df[col] = repaired_df[col].str.strip().str.strip(";")

    return repaired_df


def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="Model"):
    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
    }
    results["ROC AUC"] = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    return results


def compare_models(X, y, dataset_name, models):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    rows = []
    fitted_models = {}

    print()
    print(f"=== Model comparison for: {dataset_name} ===")
    print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

    for name, model in models.items():
        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=5,
            scoring="balanced_accuracy",
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

        row = evaluate_predictions(y_test, y_pred, y_prob, name)
        row["CV Balanced Accuracy"] = cv_scores.mean()
        rows.append(row)
        fitted_models[name] = model

        print(f"{name}: CV balanced accuracy = {cv_scores.mean():.4f}")

    results_df = pd.DataFrame(rows).sort_values(
        by=["Balanced Accuracy", "ROC AUC"], ascending=False
    ).reset_index(drop=True)
    return results_df, fitted_models, X_train, X_test, y_train, y_test

## 1. Business Understanding

Phishing websites are fake sites designed to trick users into revealing sensitive information such as passwords, banking details, or personal data.

A strong phishing-detection system is useful because it can:

- reduce cybercrime risk,
- protect users from fraudulent websites,
- support browser warnings or email security tools,
- and help organisations respond faster to suspicious links.

### Success criteria for this project

A good model should:

- correctly identify phishing websites,
- avoid too many false alarms on legitimate websites,
- generalise to more than one dataset,
- and remain understandable enough to explain to a non-technical audience.


## 2. Data Understanding — what the features mean

In machine learning, a **feature** is simply a clue that helps the model make a decision.

In this project, the clues come from:

- the **web address (URL)**,
- the **domain name**,
- the **HTML structure** of the page,
- and some **trust / reputation signals**.

### Key features from the main cleaned dataset

| Feature | Plain-English meaning | Why it can matter |
|---|---|---|
| `URLLength` | How long the web address is | Phishing URLs are often unusually long or messy |
| `DomainLength` | Length of the main domain name | Strange or very long domains can be suspicious |
| `IsDomainIP` | Whether the site uses a raw IP address instead of a domain name | Raw IPs are often less trustworthy |
| `NoOfSubDomain` | Number of subdomains in the address | Attackers often use many nested subdomains |
| `TLDLegitimateProb` | How normal or trustworthy the ending looks (such as `.com`) | Unusual endings can raise risk |
| `HasObfuscation` | Whether the URL hides information using encoded or unusual text | Obfuscation is a common phishing trick |
| `NoOfAmpersandInURL` / `NoOfEqualsInURL` | Counts of special characters in the URL | Heavily parameterised URLs can look suspicious |
| `IsHTTPS` | Whether the page uses HTTPS | HTTPS helps, but by itself does not prove safety |
| `HasPasswordField` | Whether the page contains a password field | Important because phishing pages often try to steal logins |
| `HasExternalFormSubmit` | Whether a form sends data to another location | Strong phishing warning sign |
| `NoOfPopup` / `NoOfiFrame` | Pop-ups or hidden frames on the page | Can indicate manipulation or deceptive design |
| `NoOfExternalRef`, `NoOfJS`, `NoOfImage` | How the page loads other content | Fake pages often have distinct content patterns |

### Leakage-prone features we treated carefully or removed

| Feature | What it measures | Why it was treated carefully |
|---|---|---|
| `URLSimilarityIndex` | How similar the URL is to trusted patterns | Very predictive, but may make the task unrealistically easy |
| `DomainTitleMatchScore` | Whether the page title matches the domain | Useful, but can behave like a shortcut feature |
| `URLTitleMatchScore` | Whether the title matches the URL | Same concern as above |
| `LineOfCode` | Amount of HTML code on the page | Can reflect data collection bias rather than true security |
| `HasDescription`, `HasCopyrightInfo`, `Robots`, `IsResponsive` | Website metadata / page structure clues | Helpful, but may depend on how the dataset was built |

### Key features from the external public datasets

| Feature | Plain-English meaning |
|---|---|
| `having_IP_Address` | Whether the address uses an IP instead of a normal domain |
| `URL_Length` | Whether the URL is short, medium, or long |
| `Shortining_Service` | Whether a URL shortener hides the destination |
| `having_At_Symbol` | Whether the URL contains `@`, which is often suspicious |
| `double_slash_redirecting` | Whether the URL looks like it may redirect unexpectedly |
| `Prefix_Suffix` | Whether the domain uses a suspicious hyphen pattern |
| `SSLfinal_State` | Quality of SSL / HTTPS usage |
| `Request_URL` | Whether page resources are loaded in a suspicious way |
| `URL_of_Anchor` | Whether page links point to suspicious destinations |
| `Links_in_tags` | Link behaviour inside HTML tags |
| `SFH` | Whether the form handler behaves suspiciously |
| `Submitting_to_email` | Whether form data is sent by email |
| `Abnormal_URL` | Whether the URL structure looks abnormal |
| `popUpWidnow` / `Iframe` | Whether the page relies on popups or frames |
| `age_of_domain` | How old the domain is |
| `web_traffic` | Whether the site seems well known or obscure |
| `Google_Index` | Whether Google indexes the site |
| `Statistical_report` | External report or rule-based suspicion flag |

> In short: the model is not reading meaning like a human; it is learning from many small warning signs.


In [2]:
# Load the main dataset used in the original analysis
primary_path = BASE_DIR / "data set phishing websites.csv"

df_raw = pd.read_csv(
    primary_path,
    sep=";",
    encoding="latin1",
    engine="python",
    on_bad_lines="skip",
)
df_raw = clean_column_names(df_raw)
df_raw = df_raw.loc[:, ~df_raw.columns.str.startswith("Unnamed", na=False)].copy()

target_candidates = [col for col in df_raw.columns if "label" in str(col).lower()]
target_col = target_candidates[0]

print("Raw main dataset shape:", df_raw.shape)
print("Target column:", target_col)
print("First 10 columns:", df_raw.columns[:10].tolist())
print()
print("Head of the dataset:")
print(df_raw.head())

Raw main dataset shape: (235795, 56)
Target column: label
First 10 columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb']

Head of the dataset:
     FILENAME                                 URL URLLength  \
0  521848.txt    https://www.southbankmosaics.com        31   
1   31372.txt            https://www.uni-mainz.de        23   
2  597387.txt      https://www.voicefmradio.co.uk        29   
3  554095.txt         https://www.sfnmjournal.com        26   
4  151578.txt  https://www.rewildingargentina.org        33   

                       Domain DomainLength IsDomainIP  TLD URLSimilarityIndex  \
0    www.southbankmosaics.com           24          0  com                100   
1            www.uni-mainz.de           16          0   de                100   
2      www.voicefmradio.co.uk           22          0   uk                100   
3         www.sfnmjournal.com           19          

## 3. Data Preparation

This is the step where raw data becomes usable.

We do several practical cleaning actions:

1. remove empty or broken rows,
2. find the target column,
3. keep only numeric features for modelling,
4. remove clear identifier columns such as the raw URL or file name,
5. remove **leakage-prone** columns to make the evaluation more realistic.

### Why remove leakage-prone features?

Some variables can act like shortcuts. They may allow the model to get an unrealistically high score without truly learning general phishing behaviour.

That is why the cleaned feature matrix below removes the most suspicious columns before fitting the main models.


In [3]:
# Prepare the main dataset in a more realistic way
df = df_raw.dropna(subset=["FILENAME", "URL"]).copy()
df = df[df["FILENAME"].astype(str).str.endswith(".txt", na=False)]
df = df[df["URL"].astype(str).str.startswith(("http://", "https://"), na=False)]

df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = df.dropna(subset=[target_col]).copy()

drop_cols = [target_col, "FILENAME", "URL", "Domain", "TLD", "Title"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore").copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X = X.dropna(axis=1, how="all")
mask = ~X.isna().any(axis=1)
X = X.loc[mask].copy()
y = df.loc[mask, target_col].astype(int).copy()

leakage_prone_cols = [
    "URLSimilarityIndex",
    "DomainTitleMatchScore",
    "URLTitleMatchScore",
    "LineOfCode",
    "HasDescription",
    "HasCopyrightInfo",
    "Robots",
    "IsResponsive",
]

X_clean = X.drop(columns=[c for c in leakage_prone_cols if c in X.columns], errors="ignore").copy()

print("Cleaned main feature matrix shape:", X_clean.shape)
print()
print("Class balance:")
print(y.value_counts())
print(y.value_counts(normalize=True))
print()
print("Leakage-prone columns removed:", [c for c in leakage_prone_cols if c in X.columns])
print()
print("First 25 features used in the cleaned main model:")
print(X_clean.columns[:25].tolist())

Cleaned main feature matrix shape: (135622, 42)

Class balance:
label
1    132449
0      3173
Name: count, dtype: int64
label
1    0.976604
0    0.023396
Name: proportion, dtype: float64

Leakage-prone columns removed: ['URLSimilarityIndex', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'LineOfCode', 'HasDescription', 'HasCopyrightInfo', 'Robots', 'IsResponsive']

First 25 features used in the cleaned main model:
['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LargestLineLength', 'HasTitle', 'HasFavicon', 'NoOfURLRedirect']


## 4. Modeling — compare several approaches

We do not rely on only one model.

Instead, we compare:

- **Dummy baseline** — a simple reference point that predicts the majority class
- **Random Forest** — a strong tree-based model
- **Logistic Regression** — a simpler, interpretable linear model
- **Gaussian Naive Bayes** — a lightweight statistical model

This is important because if several different models all perform well, it suggests that the dataset contains real signal rather than one model simply memorising the training data.


In [ ]:
# Baseline check: accuracy alone can be misleading on imbalanced data
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_clean, y, test_size=0.2, stratify=y, random_state=42
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_base, y_train_base)
dummy_pred = dummy.predict(X_test_base)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test_base, dummy_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test_base, dummy_pred))

# Cross-validation for the cleaned Random Forest
rf_cv = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
cv_scores = cross_val_score(rf_cv, X_clean, y, cv=5, scoring="balanced_accuracy", n_jobs=-1)
print()
print("Random Forest 5-fold balanced accuracy scores:", np.round(cv_scores, 4))
print("Mean CV balanced accuracy:", cv_scores.mean())

main_models = {
    "Random Forest": RandomForestClassifier(
random_state=42, n_jobs=-1, class_weight="balanced"
    ),
    "Logistic Regression": make_pipeline(
StandardScaler(),
LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    ),
    "Gaussian Naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
}

main_results, main_fitted_models, X_train_main, X_test_main, y_train_main, y_test_main = compare_models(
    X_clean, y, "Leakage-reduced primary dataset", main_models
)

main_results

Dummy baseline
Accuracy: 0.976589861751152
Balanced accuracy: 0.5

Random Forest 5-fold balanced accuracy scores: [0.9921 0.9961 0.989  0.9968 0.9937]
Mean CV balanced accuracy: 0.9935350608847127

=== Model comparison for: Leakage-reduced primary dataset ===
Train shape: (108497, 42) | Test shape: (27125, 42)


In [ ]:
# Confusion Matrices for All Models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Assuming you have model predictions and true labels for each model
# Example: y_true, y_pred_model1, y_pred_model2, ...

models = {
    'Model 1': y_pred_model1,
    'Model 2': y_pred_model2,
    # Add more models as needed
}

for name, y_pred in models.items():
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

In [ ]:
# Correlation Matrix
import seaborn as sns
import matplotlib.pyplot as plt

corr = X.corr()
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()
display(corr)

In [ ]:
# VIF Score Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Assuming your features are in a DataFrame called X (replace if different)
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
display(vif_data)

## Confusion Matrices for All Models
This section shows the confusion matrices for all classification models used.

## Correlation Matrix
This section displays the correlation matrix for all features.

In [ ]:
# VIF Score Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Replace X with your feature DataFrame if different
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data['feature'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

vif_df = calculate_vif(X)
display(vif_df)

In [ ]:
# Confusion Matrices for All Models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Example: y_true, y_pred_model1, y_pred_model2, ...
models = {
    'Model 1': y_pred_model1,
    'Model 2': y_pred_model2,
    # Add more models as needed
}

for name, y_pred in models.items():
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

In [ ]:
# Correlation Matrix
import seaborn as sns
import matplotlib.pyplot as plt

corr = X.corr()
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()
display(corr)

In [ ]:
# VIF Score Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Replace X with your feature DataFrame if different
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data['feature'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

vif_df = calculate_vif(X)
display(vif_df)

## VIF Score Analysis
This section calculates the Variance Inflation Factor (VIF) for all features to check for multicollinearity.

## 5. Evaluation — how to judge the results fairly

When a dataset is imbalanced, **accuracy alone can be misleading**. For example, if almost everything belongs to one class, a model can score highly just by guessing the majority class.

That is why this notebook focuses on additional metrics:

- **Balanced accuracy**: gives equal importance to both classes
- **Precision**: when the model predicts phishing, how often is it correct?
- **Recall**: how many phishing sites does the model successfully catch?
- **F1-score**: balance between precision and recall
- **ROC AUC**: how well the model ranks phishing higher than legitimate cases across thresholds

The main result from the first dataset is that the models are **very strong**, but the score was so high that extra realism checks were needed. That is why external validation was added next.


## 6. External validation on `dataset_phishing.csv`

A good way to test whether a model is truly useful is to try it on a **different dataset**.

Instead of forcing the old model onto a mismatched feature schema, we train a **new model directly on the external dataset** and compare several candidate algorithms fairly.


In [ ]:
dataset_phishing_path = BASE_DIR / "dataset_phishing.csv"
status_df = load_repaired_csv(dataset_phishing_path)

status_target = next(col for col in status_df.columns if str(col).lower() == "status")
y_status = (
    status_df[status_target]
    .astype(str)
    .str.replace(";", "", regex=False)
    .str.strip()
    .str.lower()
    .map(
        {
            "phishing": 1,
            "malicious": 1,
            "bad": 1,
            "legitimate": 0,
            "benign": 0,
            "safe": 0,
        }
    )
)

valid_mask = y_status.notna()
status_df = status_df.loc[valid_mask].copy()
y_status = y_status.loc[valid_mask].astype(int)

X_status = status_df.drop(
    columns=[c for c in [status_target, "url", "URL"] if c in status_df.columns],
    errors="ignore",
).copy()
for col in X_status.columns:
    X_status[col] = pd.to_numeric(X_status[col], errors="coerce")

X_status = X_status.replace([np.inf, -np.inf], np.nan)
X_status = X_status.dropna(axis=1, how="all")
X_status = X_status.fillna(X_status.median(numeric_only=True))

status_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced", min_samples_leaf=2
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400, random_state=42, n_jobs=-1, class_weight="balanced", min_samples_leaf=2
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=42, max_depth=8, learning_rate=0.08, max_iter=200
    ),
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    ),
}

print("dataset_phishing.csv shape:", X_status.shape)
print("Class balance:")
print(y_status.value_counts())

status_results, status_fitted_models, X_train_status, X_test_status, y_train_status, y_test_status = compare_models(
    X_status, y_status, "dataset_phishing.csv", status_models
)

status_results

## 7. External validation on `Phishing_Websites_Data.csv`

This dataset uses a different target column called **`Result`**.

In this data, the labels are encoded as `-1` and `1`, so we convert them into standard machine-learning labels:

- `-1` → `0` (legitimate)
- `1` → `1` (phishing)

We again compare several models and choose the best one using **balanced accuracy**.


In [ ]:
phishing_websites_path = BASE_DIR / "Phishing_Websites_Data.csv"
third_df = pd.read_csv(phishing_websites_path)
third_df = clean_column_names(third_df)

third_target_col = next(col for col in third_df.columns if str(col).lower() == "result")
third_y = pd.to_numeric(third_df[third_target_col], errors="coerce").replace({-1: 0, 1: 1})

valid_mask = third_y.notna()
third_df = third_df.loc[valid_mask].copy()
third_y = third_y.loc[valid_mask].astype(int)

third_X = third_df.drop(
    columns=[c for c in [third_target_col, "URL", "url"] if c in third_df.columns],
    errors="ignore",
).copy()
for col in third_X.columns:
    third_X[col] = pd.to_numeric(third_X[col], errors="coerce")

third_X = third_X.replace([np.inf, -np.inf], np.nan)
third_X = third_X.dropna(axis=1, how="all")
third_X = third_X.fillna(third_X.median(numeric_only=True))

third_models = {
    "Random Forest": RandomForestClassifier(
n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced", min_samples_leaf=2
    ),
    "Extra Trees": ExtraTreesClassifier(
n_estimators=400, random_state=42, n_jobs=-1, class_weight="balanced", min_samples_leaf=2
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
random_state=42, max_depth=8, learning_rate=0.08, max_iter=200
    ),
    "Logistic Regression": make_pipeline(
StandardScaler(),
LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    ),
}

print("Phishing_Websites_Data.csv shape:", third_X.shape)
print("Class balance:")
print(third_y.value_counts())

third_results, third_fitted_models, X_train_third, X_test_third, y_train_third, y_test_third = compare_models(
    third_X, third_y, "Phishing_Websites_Data.csv", third_models
)

third_results

## 8. Advanced evaluation: threshold tuning, error analysis, and feature importance

This section goes beyond raw accuracy.

### Why this matters

- **Threshold tuning** helps choose a better cut-off than the default `0.50`
- **Error analysis** shows where the model still makes mistakes
- **Permutation importance** shows which features matter most in practice

These steps make the project more mature and easier to justify academically.


In [ ]:
# Pick the best model from the third dataset and inspect it more closely
third_best_model_name = third_results.iloc[0]["Model"]
third_best_model = third_fitted_models[third_best_model_name]

third_y_pred = third_best_model.predict(X_test_third)
third_y_prob = (
    third_best_model.predict_proba(X_test_third)[:, 1]
    if hasattr(third_best_model, "predict_proba")
    else None
)

print("Best model selected on Phishing_Websites_Data.csv:", third_best_model_name)
print()
print("Confusion matrix:")
print(confusion_matrix(y_test_third, third_y_pred))
print()
print("Classification report:")
print(classification_report(y_test_third, third_y_pred, zero_division=0))

if third_y_prob is not None:
    # ROC + PR curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    RocCurveDisplay.from_predictions(y_test_third, third_y_prob, ax=axes[0], color="darkorange")
    axes[0].set_title(f"ROC Curve - {third_best_model_name}")

    PrecisionRecallDisplay.from_predictions(y_test_third, third_y_prob, ax=axes[1], color="darkgreen")
    axes[1].set_title(f"Precision-Recall Curve - {third_best_model_name}")
    plt.tight_layout()
    plt.show()

    # Threshold tuning
    precision_vals, recall_vals, threshold_vals = precision_recall_curve(y_test_third, third_y_prob)
    threshold_table = pd.DataFrame(
{
    "threshold": threshold_vals,
    "precision": precision_vals[:-1],
    "recall": recall_vals[:-1],
}
    )
    threshold_table["f1"] = 2 * (
threshold_table["precision"] * threshold_table["recall"]
    ) / (threshold_table["precision"] + threshold_table["recall"] + 1e-9)

    best_threshold_row = threshold_table.loc[threshold_table["f1"].idxmax()]
    best_threshold = float(best_threshold_row["threshold"])
    tuned_pred = (third_y_prob >= best_threshold).astype(int)

    print(f"Best threshold by F1: {best_threshold:.4f}")
    print(f"Precision at that threshold: {best_threshold_row['precision']:.4f}")
    print(f"Recall at that threshold: {best_threshold_row['recall']:.4f}")
    print(f"F1-score at that threshold: {best_threshold_row['f1']:.4f}")
    print()
    print("Classification report at the tuned threshold:")
    print(classification_report(y_test_third, tuned_pred, zero_division=0))

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_table["threshold"], threshold_table["precision"], label="Precision")
    plt.plot(threshold_table["threshold"], threshold_table["recall"], label="Recall")
    plt.plot(threshold_table["threshold"], threshold_table["f1"], label="F1-score")
    plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.2f}")
    plt.xlabel("Decision threshold")
    plt.ylabel("Score")
    plt.title("Threshold tuning on Phishing_Websites_Data.csv")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Error analysis: inspect the remaining mistakes
error_df = third_df.loc[X_test_third.index].copy()
error_df["actual"] = y_test_third.values
error_df["predicted"] = third_y_pred
if third_y_prob is not None:
    error_df["phishing_probability"] = third_y_prob

misclassified = error_df[error_df["actual"] != error_df["predicted"]].copy()
print()
print(f"Number of misclassified cases: {len(misclassified)} out of {len(error_df)}")
print("Sample of misclassified rows:")
print(misclassified.head(10))

# Permutation importance: how much each feature matters in practice
perm = permutation_importance(
    third_best_model,
    X_test_third,
    y_test_third,
    n_repeats=10,
    random_state=42,
    scoring="balanced_accuracy",
    n_jobs=-1,
)

perm_df = pd.DataFrame(
    {
"Feature": X_test_third.columns,
"Importance": perm.importances_mean,
    }
).sort_values("Importance", ascending=False)

print()
print("Top 15 permutation importances:")
print(perm_df.head(15))

plt.figure(figsize=(10, 6))
top_features = perm_df.head(15).iloc[::-1]
plt.barh(top_features["Feature"], top_features["Importance"], color="teal")
plt.title("Top 15 Permutation Importances - Best External Model")
plt.xlabel("Mean decrease in balanced accuracy")
plt.tight_layout()
plt.show()

## 9. Deployment — the phishing prototype app

A project becomes much stronger when it moves beyond analysis and includes a usable demo.

This workspace now includes a simple **Streamlit prototype** called:

- `phishing_prototype_app.py`

The app:

- trains a phishing model on `Phishing_Websites_Data.csv`,
- lets a user enter phishing indicators manually,
- predicts **phishing vs legitimate**,
- shows a **probability and risk band**,
- and can score a whole CSV file in batch mode.

### How to run the prototype locally

Open PowerShell and run:

```powershell
Set-Location "c:\Users\nanaa\OneDrive\Naive Bayes Nana Anderson"
.\.venv\Scripts\python.exe -m streamlit run phishing_prototype_app.py
```

---

## Final conclusion

The project shows that phishing websites can be detected with **very strong performance**, especially when the model is trained on a dataset whose features match the task well.

At the same time, the project also shows an important scientific lesson:

> **Very high internal accuracy is not enough by itself. Real confidence comes from leakage checks, external validation, threshold tuning, and error analysis.**

Therefore, the best conclusion is **not** that the model is “perfect,” but that it is **highly effective, well-tested, and practically useful**.
